In [1]:
import sys, os
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

# --- robust project root detection ---
CWD = Path.cwd()
PROJECT_ROOT = next((p for p in [CWD] + list(CWD.parents) if (p / "preprocessing").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find PROJECT_ROOT (no 'preprocessing/' directory found in parents).")

# Make imports work + ensure relative paths are consistent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("CWD:", CWD)
print("PROJECT_ROOT:", PROJECT_ROOT)

import pandas as pd

from analysis.metabolite_level_analysis import (analyse_with_logistic_regression, calculate_odds_ratio)

from analysis.config import make_default_paths
# dataset root
DATA_DIR = PROJECT_ROOT / "ROSMAP-AD"
paths = make_default_paths(DATA_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
metabolite_levels_file_path = paths.metabolite_levels_file_path

from preprocessing import (
    MetaboliteLevelsPreprocessorFactory,
    PathwayPreprocessorFactory,
    ReactionPreprocessorFactory,
    MetabolitePathwayCombinedPreprocessorFactory,
    MetaboliteReactionCombinedPreprocessorFactory,
    TranscriptomicsPreprocessorFactory,
    TranscriptomicsMetabolitePreprocessorFactory,
    TranscriptomicsPathwayPreprocessorFactory,
    TranscriptomicsReactionPreprocessorFactory
)

CWD: /home/bioinformatics/Desktop/alzheimer_analysis/notebooks
PROJECT_ROOT: /home/bioinformatics/Desktop/alzheimer_analysis
PROJECT_ROOT: /home/bioinformatics/Desktop/alzheimer_analysis
DATA_DIR: /home/bioinformatics/Desktop/alzheimer_analysis/ROSMAP-AD


In [2]:
prs_combined = paths.prs_combined_path
pathway_file_path = paths.pathway_file_path
reaction_file_path = paths.reaction_file_path
metabolite_level_pathway_combined_file_path = paths.metabolite_level_pathway_combined_file_path
metabolite_level_reaction_combined_file_path = paths.metabolite_level_reaction_combined_file_path
x_pathway_path = paths.x_pathway_path
x_reaction_path = paths.x_reaction_path
transcriptomics_path = paths.transcriptomics_path
transcriptomics_metabolites_combined_path = paths.transcriptomics_metabolites_combined_path
transcriptomics_pathway_combined_path = paths.transcriptomics_pathway_combined_path
transcriptomics_reaction_combined_path = paths.transcriptomics_reaction_combined_path
transcriptomics_path = paths. transcriptomics_path

factory_list = [
    MetaboliteLevelsPreprocessorFactory(metabolite_levels_file_path),
    PathwayPreprocessorFactory(metabolite_levels_file_path, pathway_file_path, x_pathway_path),
    ReactionPreprocessorFactory(metabolite_levels_file_path, reaction_file_path, x_reaction_path),
    MetabolitePathwayCombinedPreprocessorFactory(metabolite_levels_file_path, metabolite_level_pathway_combined_file_path, x_pathway_path),
    MetaboliteReactionCombinedPreprocessorFactory(metabolite_levels_file_path, metabolite_level_reaction_combined_file_path, x_reaction_path),
    TranscriptomicsPreprocessorFactory(transcriptomics_path),
    TranscriptomicsMetabolitePreprocessorFactory(metabolite_levels_file_path, transcriptomics_metabolites_combined_path, transcriptomics_path),
    TranscriptomicsPathwayPreprocessorFactory(metabolite_levels_file_path, transcriptomics_pathway_combined_path, transcriptomics_path, x_pathway_path),
    TranscriptomicsReactionPreprocessorFactory(metabolite_levels_file_path, transcriptomics_reaction_combined_path, transcriptomics_path, x_reaction_path)
]

odds_ratio_list = []

for factory in factory_list:
    preprocessor = factory.create_preprocessor()
    y, risk_scores, sample_ids_ = analyse_with_logistic_regression(preprocessor)
    odds_ratio = calculate_odds_ratio(risk_scores, y)
    odds_ratio_list.append({preprocessor.name: odds_ratio})

for odds_ratio in odds_ratio_list:
    name = list(odds_ratio.keys())[0]
    value = list(odds_ratio.values())[0]
    print(f"Preprocessor: {name}, Odds Ratio: {value}")


Preprocessor: Metabolite Levels, Odds Ratio: 164.88189009297287
Preprocessor: Pathway Diff Scores, Odds Ratio: 56.682458699398495
Preprocessor: Reaction Diff Scores, Odds Ratio: 58.795754923728836
Preprocessor: Metabolite Levels Pathway Combined, Odds Ratio: 183.26620537403602
Preprocessor: Metabolite Levels Reaction Combined, Odds Ratio: 80.2982198329898
Preprocessor: Transcriptomics, Odds Ratio: 50.22061083984625
Preprocessor: Transcriptomics Metabolite Levels Combined, Odds Ratio: 53.46536086065514
Preprocessor: Transcriptomics Pathway Diff Combined, Odds Ratio: 51.78780988810083
Preprocessor: Transcriptomics Reaction Diff Combined, Odds Ratio: 38.74277075134126


In [3]:
print("Preprocessor".ljust(50) + "Odds Ratio")
print("-" * 40)
for odds_ratio in odds_ratio_list:
    name = list(odds_ratio.keys())[0]
    value = list(odds_ratio.values())[0]
    print(f"{name}".ljust(50) + f"{value}")

Preprocessor                                      Odds Ratio
----------------------------------------
Metabolite Levels                                 164.88189009297287
Pathway Diff Scores                               56.682458699398495
Reaction Diff Scores                              58.795754923728836
Metabolite Levels Pathway Combined                183.26620537403602
Metabolite Levels Reaction Combined               80.2982198329898
Transcriptomics                                   50.22061083984625
Transcriptomics Metabolite Levels Combined        53.46536086065514
Transcriptomics Pathway Diff Combined             51.78780988810083
Transcriptomics Reaction Diff Combined            38.74277075134126


In [6]:
factory_ = MetaboliteLevelsPreprocessorFactory(metabolite_levels_file_path)
my_preprocessor = factory_.create_preprocessor()
y, risk_scores, sample_ids = analyse_with_logistic_regression(my_preprocessor)
print(sample_ids)
prs_df = pd.read_csv("ROSMAP-AD/raw/prs_scores.csv")
print(prs_df)

0      ROS15176592
1      ROS15196848
2      MAP17929065
3      ROS20532115
4      ROS20674902
          ...     
265    MAP63188799
266    MAP66924745
267    MAP71648351
268    MAP76755449
269    MAP92023910
Name: Sample ID, Length: 268, dtype: str
       sample_id     score
0    MAP12788806 -0.254295
1    MAP13485298 -0.603982
2    MAP13747771 -0.254957
3    MAP14184286 -0.006826
4    MAP14452889 -0.135003
..           ...       ...
377  ROS60269051 -0.273474
378  ROS70421820 -0.026957
379  ROS75142238 -0.221273
380  ROS97411796 -0.244321
381  ROS97929179 -0.039422

[382 rows x 2 columns]
